In [ ]:
import math
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

from itertools import count
from importnb import Notebook
from typing import List, Dict, Any

with Notebook():
    from LabLatencyModel import LatencyModel, MultiDULatencyModel
    from LabCacheEngine import CacheEngineEnv
    from LabUserRequest import UserRequestEvents
    from LabPrefetchScheduler import PrefetchScheduler

In [ ]:
class EnvWrapper(gym.Env):
    """
    Simple wrapper that delegates all calls to an inner env.
    Subclass this to create your own wrappers.
    """

    metadata = {"render.modes": []}

    def __init__(
        self, 
        cfg: Any,
        n: int,
        m: int,
        n_layers: int,
        lam: float,
        theta: float,
        users_env: None,
        du_caches: None,
        mec_cache: None,
        latency_model: None,
        prefetch_fn: None,
        reward_fn: None,
        max_steps: int = 10000,
        *,
        step_duration_s=1.0
    ):
        super().__init__()

        self.step_count = 0
        self.step_duration_s = step_duration_s
        self.max_steps = max_steps

        self.n = n  # number of tiles per row/column
        self.m = m  # number of tiles per row/column
        self.n_layers = n_layers  # number of layers (base + enhancement)
        
        self.gain_if_prefetched = 1.0
        self.loss_if_not_prefetched = -1.0

        self.theta = theta
        self.lam = lam

        self.users_env = users_env
        self.du_caches = du_caches
        self.mec_cache = mec_cache
        self.latency_model = latency_model

        self.prefetch_fn = prefetch_fn or (lambda cache, action: cache.drl_prefetching(action))
        self.reward_fn = reward_fn or (lambda info: info.get('reward_per_user', {}).get(info.get('current_user', -1), 0.0))

        # History holders
        self.users_reward = {
            u: [] for u in range(self.users_env.n_users)
        }
        self.users_psnr = {
            u: [] for u in range(self.users_env.n_users)
        }
        self.total_gop_requests_per_user = {
            u: 0 for u in range(self.users_env.n_users)
        }
        
        self.scheduler = PrefetchScheduler(
            R_M_D=self.latency_model.R_M_D,
            R_C_M=self.latency_model.R_C_M,
            U=self.latency_model.max_U,
            step_duration_s=self.step_duration_s
        )

    # ─────────────────────────────────────────────────────────────────────────
    # Internal Helpers
    # ─────────────────────────────────────────────────────────────────────────
    def _make_ready_bitmaps(self, du_planned, mec_planned):
        du_ready = None
        mec_ready = None

        if du_planned:
            du_ready = []
            for du_idx, bm in enumerate(du_planned):
                cache_key = f"DU:{du_idx}"
                du_ready.append(self.scheduler.materialize_ready_bitmap(cache_key, bm))

        if mec_planned is not None:
            mec_ready = self.scheduler.materialize_ready_bitmap("MEC", mec_planned)

        return du_ready, mec_ready

    def _missing_items(self, req):
        missing = 5 * [0]

        layer = 0
        for tile in req["tiles"]:
            du_hit  = tile["events"]["alpha_p_u"]
            mec_hit = tile["events"]["alpha_M_u"]
            layer   = tile["layer"]
            
            if not du_hit and not mec_hit and layer == 0:
                missing[0] = 1
                break

        layer = 1
        for i, tile in enumerate(req["viewport"]):
            mec_hit = self.mec_cache.check_tile_in_cache(req["video"], layer,tile, 0)

            if not mec_hit:
                missing[i + 1] = 1

        return missing

    def _process_prefetch_actions(self, req, action, du_plan, mec_plan):

        user = req["u"]
        video = req["video"]
        viewport = req["viewport"]

        missing = self._missing_items(req)

        for idx, is_missing in enumerate(missing):
            if not is_missing:
                continue

            base_slot = self.mec_cache.get_video_cache_idx(video)
            if idx > 0 and base_slot == -1:
                break

            message = {
                "video": video,
                "tiles": [] if idx == 0 else [viewport[idx - 1]],
                "base_layer_req": (idx == 0),
                "action_idx": action[idx],
            }

            # Apply to MEC cache
            mec_plan = self.prefetch_fn(self.mec_cache, message)

        return [], du_plan, mec_plan

    def _compute_cache_hits(self, req):
        video = req["video"]
        viewport = req["viewport"]

        video_cache_index = self.mec_cache.policy.video_idx
        tile_cache_index = self.mec_cache.policy.tile_idx

        base_hit = 1 if video in video_cache_index else 0
        
        video_idx = self.mec_cache.get_video_cache_idx(video)

        if video_idx != -1:
            cached_tiles = tile_cache_index[video_idx]
            enh_hit = [1 if tile in cached_tiles else 0 for tile in viewport]
        else:
            enh_hit = [0, 0, 0, 0]
    
        return dict(
            base_layer_hits=12 * base_hit,
            enh_layer_hits=sum(enh_hit),
            base_layer_misses=12 * base_hit,
            enh_layer_misses=4 - sum(enh_hit)
        )

    # ─────────────────────────────────────────────────────────────────────────
    # Gym Environment API
    # ─────────────────────────────────────────────────────────────────────────
    def step(self, action, req=None):
        info = {}

        # -------------------------------------------------------
        # 1: Apply prefetching for all actions this step
        # -------------------------------------------------------
        du_plan = self.du_caches if len(self.du_caches) > 0 else None
        mec_plan = self.mec_cache.get_cache_bitmap() if self.mec_cache else None

        _, du_plan, mec_plan = self._process_prefetch_actions(
            req, action, du_plan, mec_plan
        )
            
        # Immediate user request
        req = self.users_env.get_next_request(du_plan, mec_plan)

        # -------------------------------------------------------
        # 2. Remaining users requests
        # -------------------------------------------------------
        info["user_request"] = req

        # -------------------------------------------------------
        # 3. Cache stats (HIT / MISS)
        # -------------------------------------------------------
        info.update(self._compute_cache_hits(req))

        done = self.users_env.all_users_done() or (self.step_count >= self.max_steps - 1)
        self.step_count += 1

        return {}, 0.0, done, info

    # ---------------------------------------------------------
    #  HIT / MISS STATS
    # ---------------------------------------------------------
    def compute_cache_stats(self, reqs):

        base_hits = 0
        enh_hits  = 0
        base_miss = 0
        enh_miss  = 0

        for req in reqs:
            for tile in req["tiles"]:
                l = tile["layer"]
                hit = tile["events"]["alpha_p_u"] or tile["events"]["alpha_M_u"]

                if l == 0:
                    base_hits += int(hit)
                    base_miss += int(not hit)
                else:
                    enh_hits += int(hit)
                    enh_miss += int(not hit)

        return dict(
            base_layer_hits=base_hits,
            enh_layer_hits=enh_hits,
            base_layer_misses=base_miss,
            enh_layer_misses=enh_miss
        )

    # ---------------------------------------------------------
    # SAMPLE ACTION
    # ---------------------------------------------------------
    def sample_action(self):
        tiles = np.zeros(self.n * self.m, dtype=int)
        c = self.n // 2
        if self.n % 2 == 1:
            center_idx = c * self.n + c
            tiles[center_idx] = 1
        else:
            centers = [(c-1, c-1), (c-1, c), (c, c-1), (c, c)]
            for x, y in centers:
                tiles[y * self.n + x] = 1

        return {
            'video': np.random.randint(0, self.users_env.n_videos),
            'gop': np.random.randint(0, self.users_env.n_gops),
            'tiles': tiles.tolist()
        }, c * self.n + c

    # ---------------------------------------------------------
    # RESET
    # ---------------------------------------------------------
    def reset(self, **kwargs):
        self.step_count = 0
        
        self.users_reward = {
            u: [] for u in range(self.users_env.n_users)
        }
        self.users_psnr = {
            u: [] for u in range(self.users_env.n_users)
        }
        self.total_gop_requests_per_user = {
            u: 0 for u in range(self.users_env.n_users)
        }

        _, info_users = self.users_env.reset(**kwargs)

        info_cache_mec = self.mec_cache.reset(**kwargs)[1] if self.mec_cache else {}

        req = self.users_env.get_next_request(None, None)

        info_cache = {
            **info_cache_mec,
            **info_users,
            "user_request": req
        }

        # Reset scheduler's time and availability
        self.scheduler.now_s = 0.0
        self.scheduler.availability = {}

        return None, info_cache